# 01 - Data Ingestion

This notebook performs the initial ingestion of raw UK electricity demand data files.

The workflow includes:

- Reading multiple raw CSV files
- Combining datasets
- Performing initial inspection
- Preparing the data for downstream transformation and validation

In [1]:
%reset -f

import pandas as pd
from pathlib import Path

In [3]:
raw_data_path = Path("../Data/Raw")
csv_files = list(raw_data_path.glob("*.csv"))
csv_files

[WindowsPath('../Data/Raw/demanddata_2021.csv'),
 WindowsPath('../Data/Raw/demanddata_2022.csv'),
 WindowsPath('../Data/Raw/demanddata_2023.csv'),
 WindowsPath('../Data/Raw/demanddata_2024.csv'),
 WindowsPath('../Data/Raw/demanddata_2025.csv'),
 WindowsPath('../Data/Raw/demanddata_2026.csv')]

## Read and Combine CSV Files

In [7]:
df_list = []

for file in csv_files:
    df = pd.read_csv(file)
    df_list.append(df)

combined_df = pd.concat(df_list, ignore_index=True)
combined_df.head()

,SETTLEMENT_DATE,SETTLEMENT_PERIOD,ND,TSD,ENGLAND_WALES_DEMAND,EMBEDDED_WIND_GENERATION,EMBEDDED_WIND_CAPACITY,EMBEDDED_SOLAR_GENERATION,EMBEDDED_SOLAR_CAPACITY,NON_BM_STOR,...,IFA2_FLOW,BRITNED_FLOW,MOYLE_FLOW,EAST_WEST_FLOW,NEMO_FLOW,NSL_FLOW,ELECLINK_FLOW,VIKING_FLOW,GREENLINK_FLOW,SCOTTISH_TRANSFER
0,01-JAN-2021,1,28354,28969,26130,1158,6527,0,13471,0,...,-1,0,215,203,999,0,0,0,0,NaN
1,01-JAN-2021,2,28501,29114,26281,1208,6527,0,13471,0,...,-1,0,359,203,999,0,0,0,0,NaN
2,01-JAN-2021,3,27759,28376,25557,1202,6527,0,13471,0,...,-1,0,362,202,999,0,0,0,0,NaN
3,01-JAN-2021,4,26912,27749,24792,1226,6527,0,13471,0,...,-1,0,361,203,1000,0,0,0,0,NaN
4,01-JAN-2021,5,26004,27178,23933,1193,6527,0,13471,0,...,-1,0,304,203,1000,0,0,0,0,NaN


## Initial Data Inspection

In [10]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89472 entries, 0 to 89471
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   SETTLEMENT_DATE            89472 non-null  object 
 1   SETTLEMENT_PERIOD          89472 non-null  int64  
 2   ND                         89472 non-null  int64  
 3   TSD                        89472 non-null  int64  
 4   ENGLAND_WALES_DEMAND       89472 non-null  int64  
 5   EMBEDDED_WIND_GENERATION   89472 non-null  int64  
 6   EMBEDDED_WIND_CAPACITY     89472 non-null  int64  
 7   EMBEDDED_SOLAR_GENERATION  89472 non-null  int64  
 8   EMBEDDED_SOLAR_CAPACITY    89472 non-null  int64  
 9   NON_BM_STOR                89472 non-null  int64  
 10  PUMP_STORAGE_PUMPING       89472 non-null  int64  
 11  IFA_FLOW                   89472 non-null  int64  
 12  IFA2_FLOW                  89472 non-null  int64  
 13  BRITNED_FLOW               89472 non-null  int

## Create Datetime Column

In [15]:
combined_df["SETTLEMENT_DATE"] = pd.to_datetime(
    combined_df["SETTLEMENT_DATE"],
    format="%d-%b-%Y"
)

combined_df["DATETIME"] = (
    combined_df["SETTLEMENT_DATE"]
    + pd.to_timedelta(
        (combined_df["SETTLEMENT_PERIOD"] - 1) * 30,
        unit="m")
)

combined_df[[
    "SETTLEMENT_DATE",
    "SETTLEMENT_PERIOD",
    "DATETIME"
]].head()

,SETTLEMENT_DATE,SETTLEMENT_PERIOD,DATETIME
0,2021-01-01,1,2021-01-01 00:00:00
1,2021-01-01,2,2021-01-01 00:30:00
2,2021-01-01,3,2021-01-01 01:00:00
3,2021-01-01,4,2021-01-01 01:30:00
4,2021-01-01,5,2021-01-01 02:00:00


## Missing Value Analysis

In [18]:
missing_values = combined_df.isnull().sum()
missing_values[missing_values > 0]

SCOTTISH_TRANSFER    35040
dtype: int64

## Handle Missing Values

The `SCOTTISH_TRANSFER` column contains missing values.

For the purposes of this project, missing values will be replaced with `0` to maintain dataset consistency for downstream processing and cloud loading.

In [21]:
combined_df["SCOTTISH_TRANSFER"] = (
    combined_df["SCOTTISH_TRANSFER"]
    .fillna(0)
)

combined_df["SCOTTISH_TRANSFER"].isnull().sum()

0

## Save Processed Dataset & Confirm Output

In [26]:
processed_path = Path("../Data/Processed")
processed_path.mkdir(parents=True, exist_ok=True)

combined_df.to_csv(
    processed_path / "combined_energy_demand.csv",
    index=False
)

print("Processed dataset saved successfully.")

processed_file = processed_path / "combined_energy_demand.csv"
processed_file.exists()

Processed dataset saved successfully.


True